In [1]:
import pandas as pd
import pickle
import webdataset as wds

In [2]:
checkpoints = {
    "protein_mpnn": "./model_params/proteinmpnn_v_48_020.pt",
    "ligand_mpnn": "./model_params/ligandmpnn_v_32_010_25.pt",
    "per_residue_label_membrane_mpnn": "./model_params/per_residue_label_membrane_mpnn_v_48_020.pt",
    "global_label_membrane_mpnn": "./model_params/global_label_membrane_mpnn_v_48_020.pt",
    "soluble_mpnn": "./model_params/solublempnn_v_48_020.pt",  
    'ligand_mpnn_retrained': "/home/fosterb/LigandMPNN/model_params/ligandmpnn_default_nomixed_cont/model_weights/epoch_best.pt",
    'ligand_mpnn_potts': "/home/fosterb/LigandMPNN/model_params/ligandmpnn_potts_nomixed_cont/model_weights/epoch_best.pt",
    'ligand_mpnn_potts_v2': "/mnt/shared3/fosterb/pmpnn_runs/ligandmpnn_potts_4enc/model_weights/epoch_best.pt"
}

In [18]:
restype_3to1 = {
        "ALA": "A",
        "ARG": "R",
        "ASN": "N",
        "ASP": "D",
        "CYS": "C",
        "GLN": "Q",
        "GLU": "E",
        "GLY": "G",
        "HIS": "H",
        "ILE": "I",
        "LEU": "L",
        "LYS": "K",
        "MET": "M",
        "PHE": "F",
        "PRO": "P",
        "SER": "S",
        "THR": "T",
        "TRP": "W",
        "TYR": "Y",
        "VAL": "V",
    }
from Bio.PDB import PDBParser, PPBuilder, Polypeptide
import pandas as pd

def aa_or_x(residue):
    """Return 1-letter AA code or 'X' for non-canonical residues."""
    try:
        aa = Polypeptide.three_to_one(residue.get_resname())
    except KeyError:
        aa = "X"
    return aa

def generate_mut_dataframe(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_path)
    model = structure[0]

    # Identify protein chains (chains containing at least one residue)
    protein_chains = []
    for chain in model:
        for res in chain:
            if res.id[0] == " ":  # exclude heteroatoms, water
                protein_chains.append(chain.id)
                break
    protein_chains = sorted(set(protein_chains))

    # Build per-chain sequence using AA=X for noncanonical residues
    chain_sequences = {}
    for chain in model:
        if chain.id not in protein_chains:
            continue
        seq = []
        for res in chain:
            if res.id[0] != " ":
                pass
                # continue
            seq.append(aa_or_x(res))
        chain_sequences[chain.id] = "".join(seq)

    # Standard AA list
    aas = list("ACDEFGHIKLMNPQRSTVWY")
    WT_name = os.path.basename(pdb_path).split('.')[0]
    rows = []
    for chain_id, sequence in chain_sequences.items():
        for pos, wt_res in enumerate(sequence, start=0):

            for mut in aas:
                if mut == wt_res:
                    continue

                mut_type = f"{wt_res}{pos}{mut}"
                sig = f"{WT_name}_{chain_id}_{mut_type}"

                rows.append({
                    "WT_name": WT_name,
                    "mut_type": mut_type,
                    "ddG_ML": 0,
                    "chain": chain_id,
                    "chain_list": protein_chains,
                    "sig": sig
                })

    return pd.DataFrame(rows)


In [ ]:
## Megascale loading
test_df = pd.read_csv('/mnt/shared/fosterb/rocklin_data_2022/mega_single.csv')
with open('/mnt/shared/fosterb/ThermoMPNN/dataset_splits/mega_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
pdbs = splits['test']
megascale_settings = {}
megascale_settings['pdbs'] = splits['test']
test_df = test_df[test_df['WT_name'].isin(pdbs)]
test_df['adj_pos'] = [';'.join([  str(int(pos[1:-1]) - 1) ]) for pos in test_df['mut_type']]
megascale_settings['test_df'] = test_df
megascale_settings['raw_data_dir'] = '/mnt/shared/fosterb/rocklin_data_2022/AlphaFold_model_PDBs' ## Directory containing .pdb files
megascale_settings['offset'] = False
megascale_settings['split'] = False

## Megascale-D loading
test_df = pd.read_csv('/mnt/shared/fosterb/rocklin_data_2022/mega_double.csv')
with open('/mnt/shared/fosterb/ThermoMPNN/dataset_splits/mega_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
pdbs = splits['test']
megascale_double_settings = {}
megascale_double_settings['pdbs'] = splits['test']
test_df = test_df[test_df['WT_name'].isin(pdbs)]
test_df['adj_pos'] = [';'.join([  str(int(pos_part[1:-1]) - 1) for pos_part in pos.split(';')]) for pos in test_df['mut_type']]
megascale_double_settings['test_df'] = test_df
megascale_double_settings['raw_data_dir'] = '/mnt/shared/fosterb/rocklin_data_2022/AlphaFold_model_PDBs' ## Directory containing .pdb files
megascale_double_settings['offset'] = False
megascale_double_settings['split'] = False

## Megascale-D mean loading
test_df = pd.read_csv('/mnt/shared/fosterb/rocklin_data_2022/mega_double_mean.csv')
with open('/mnt/shared/fosterb/ThermoMPNN/dataset_splits/mega_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
pdbs = splits['test']
megascale_double_mean_settings = {}
megascale_double_mean_settings['pdbs'] = splits['test']
test_df = test_df[test_df['WT_name'].isin(pdbs)]
test_df['adj_pos'] = [';'.join([  str(int(pos_part[1:-1]) - 1) for pos_part in pos.split(';')]) for pos in test_df['mut_type']]
megascale_double_mean_settings['test_df'] = test_df
megascale_double_mean_settings['raw_data_dir'] = '/mnt/shared/fosterb/rocklin_data_2022/AlphaFold_model_PDBs' ## Directory containing .pdb files
megascale_double_mean_settings['offset'] = False
megascale_double_mean_settings['split'] = False

## Fireprot loading
test_df = pd.read_csv('/mnt/shared/fosterb/fireprot_data/fireprot_homologue_free_sort_ligandmpnn.csv')
if 'adj_pos' in test_df.columns:
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)
fireprot_settings = {
    'test_df': test_df,
    'pdbs': test_df['WT_name'].unique(),
    'raw_data_dir': '/mnt/shared/fosterb/fireprot_data/pdbs',
    'offset': False,
    'split': False
}

## Covid loading
test_df = pd.read_csv('/mnt/shared/fosterb/covid_data/covid_bind.csv')
test_df = test_df[~test_df['mut_type'].str.contains(r'\*', regex=True)]
test_df = test_df[test_df['adj_pos'] <= 193]
if 'adj_pos' in test_df.columns: ## If adjusted position lists are provided, they need to be converted into lists of ints
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)    
covid_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/covid_data/raw_pdb',
    'offset': True,
    'split': False
}

## Covid split loading
test_df = pd.read_csv('/mnt/shared/fosterb/covid_data/covid_bind.csv')
test_df = test_df[~test_df['mut_type'].str.contains(r'\*', regex=True)]
test_df = test_df[test_df['adj_pos'] <= 193]
if 'adj_pos' in test_df.columns: ## If adjusted position lists are provided, they need to be converted into lists of ints
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)    
covid_split_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/covid_data/raw_pdb',
    'offset': True,
    'split': True
}

## Covid stability loading
test_df = pd.read_csv('/mnt/shared/fosterb/covid_data/covid_stability.csv')
test_df = test_df[~test_df['mut_type'].str.contains(r'\*', regex=True)]
test_df = test_df[test_df['adj_pos'] <= 193]
if 'adj_pos' in test_df.columns: ## If adjusted position lists are provided, they need to be converted into lists of ints
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)    
covid_stability_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/covid_data/raw_pdb',
    'offset': True,
    'split': False
}

## Protabank loading
test_df = pd.read_csv('/mnt/shared/fosterb/protabank_data/1jmq_data_thermo.csv')
if 'adj_pos' in test_df.columns: ## If adjusted position lists are provided, they need to be converted into lists of ints
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)  
protabank_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/protabank_data/pdbs',
    'offset': False,
    'split': False
}

## PPB loading
test_df = pd.read_csv('/mnt/shared/fosterb/PPB/ppb_ligandmpnn.csv')
if 'adj_pos' in test_df.columns: ## If adjusted position lists are provided, they need to be converted into lists of ints
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)  
test_df = test_df[~test_df['WT_name'].isin(['3R9A_A_B_C.pdb', '4CPA_A_I.pdb'])]
ppb_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/PPB/pdbs',
    'offset': True,
    'three_plus_chains': True,
    'split': False
}

## S669 loading
test_df = pd.read_csv('/mnt/shared/fosterb/S669_data/S669_thermo_parsed.csv')
if 'adj_pos' in test_df.columns: ## If adjusted position lists are provided, they need to be converted into lists of ints
    test_df['adj_pos'] = test_df['adj_pos'].astype(str)  
S669_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/S669_data/pdbs',
    'offset': True,
    'three_plus_chains': True,
    'split': False
}

## Bcl2 loading
cols = ['inp.pyd']
wds_dataset = wds.WebDataset('/mnt/shared/fosterb/bcl2_data/bcl2.wds').decode().to_tuple(*cols)
chain_lens = {}
for data in wds_dataset:
    chain_lens[data[0]['pdb']] = data[0]['chain_lens'][0]

test_df = pd.read_csv('/mnt/shared/fosterb/bcl2_data/bcl2_data_all.csv')
test_df = test_df.rename(columns={'protein': 'WT_name', 'mutation': 'mut_type', 'ener': 'ddG_ML', })
test_df['chain_list'] = [['A', 'B']] * len(test_df)
adj_poses = []
for pos, WT_name in zip(test_df['pos'], test_df['WT_name']):
    adj_pos = ";".join([ str( int(p) - chain_lens[WT_name]  ) for p in pos.split(';') ])
    adj_poses.append(adj_pos)
test_df['adj_pos'] = adj_poses
test_df['sig'] = ['_'.join([WT_name, chain, mut_type]) for WT_name, chain, mut_type in zip(test_df['WT_name'], test_df['chain'], test_df['mut_type'])]
bcl2_settings = {
    'test_df': test_df,
    'pdbs': test_df['WT_name'].unique(),
    'raw_data_dir': '/mnt/shared/fosterb/bcl2_data/raw_pdb',
    'offset': True,
    'split': False
}

## Sun (prot-lig) loading
test_df = pd.read_csv('/mnt/shared/fosterb/protein_ligand_data/Sun_data_ligandmpnn.csv')
sun_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/protein_ligand_data/pdbs',
    'offset': False,
    'three_plus_chains': True,
    'split': False
}

## PremPLI (prot-lig) loading
import numpy as np
test_df = pd.read_csv('/mnt/shared/fosterb/protein_ligand_data/PremPLI_filt2.csv')
test_df['adj_pos'] = test_df['adj_pos'].astype(str) 
prem_settings = {
    'test_df': test_df,
    'pdbs': np.unique(test_df['WT_name'].values),
    'raw_data_dir': '/mnt/shared/fosterb/protein_ligand_data/pdbs',
    'offset': False,
    'three_plus_chains': True,
    'split': False
}

## cdc28 loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/cdc28.pdb')
cdc28_settings = {}
cdc28_settings['pdbs'] = ['cdc28']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
cdc28_settings['test_df'] = test_df
cdc28_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
cdc28_settings['offset'] = False
cdc28_settings['split'] = False

## cdc28_atp loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/cdc28_atp.pdb')
test_df['chain_list'] = [['A', 'C']] * len(test_df)
cdc28_atp_settings = {}
cdc28_atp_settings['pdbs'] = ['cdc28_atp']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
cdc28_atp_settings['test_df'] = test_df
cdc28_atp_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
cdc28_atp_settings['offset'] = False
cdc28_atp_settings['split'] = False

## cdc28_ligand loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/cdc28_ligand.pdb')
test_df['chain_list'] = [['A', 'B']] * len(test_df)
cdc28_ligand_settings = {}
cdc28_ligand_settings['pdbs'] = ['cdc28_ligand']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
cdc28_ligand_settings['test_df'] = test_df
cdc28_ligand_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
cdc28_ligand_settings['offset'] = False
cdc28_ligand_settings['split'] = False

## cdc28_only loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/cdc28_ligand.pdb')
test_df['chain_list'] = [['A']] * len(test_df)
cdc28_only_settings = {}
cdc28_only_settings['pdbs'] = ['cdc28_ligand']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
cdc28_only_settings['test_df'] = test_df
cdc28_only_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
cdc28_only_settings['offset'] = False
cdc28_only_settings['split'] = False

## swe1_atp loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/swe1_atp.pdb')
test_df['chain_list'] = [['A', 'B']] * len(test_df)
swe1_atp_settings = {}
swe1_atp_settings['pdbs'] = ['swe1_atp']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
swe1_atp_settings['test_df'] = test_df
swe1_atp_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
swe1_atp_settings['offset'] = False
swe1_atp_settings['split'] = False

## swe1_ligand loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/swe1_ligand.pdb')
test_df['chain_list'] = [['A', 'B']] * len(test_df)
swe1_ligand_settings = {}
swe1_ligand_settings['pdbs'] = ['swe1_ligand']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
swe1_ligand_settings['test_df'] = test_df
swe1_ligand_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
swe1_ligand_settings['offset'] = False
swe1_ligand_settings['split'] = False

## swe1_only loading
test_df = generate_mut_dataframe('/mnt/shared/fosterb/Joel_phosphate/pdbs/swe1_ligand.pdb')
test_df['chain_list'] = [['A']] * len(test_df)
swe1_only_settings = {}
swe1_only_settings['pdbs'] = ['swe1_ligand']
test_df['adj_pos'] = [';'.join([str(int(pos[1:-1])) ]) for pos in test_df['mut_type']]
swe1_only_settings['test_df'] = test_df
swe1_only_settings['raw_data_dir'] = '/mnt/shared/fosterb/Joel_phosphate/pdbs/' ## Directory containing .pdb files
swe1_only_settings['offset'] = False
swe1_only_settings['split'] = False

all_dataset_settings = {
    'megascale': megascale_settings,
    'megascale_double': megascale_double_settings,
    'megascale_double_mean': megascale_double_mean_settings,
    'fireprot': fireprot_settings,
    'covid': covid_settings,
    'covid_split': covid_split_settings,
    'covid_stability': covid_stability_settings,
    'protabank': protabank_settings,
    'ppb': ppb_settings,
    'bcl2': bcl2_settings,
    'S669': S669_settings,
    'Sun': sun_settings,
    'prem': prem_settings,
    'cdc28': cdc28_settings,
    'cdc28_atp': cdc28_atp_settings,
    'cdc28_ligand': cdc28_ligand_settings,
    'cdc28_only': cdc28_only_settings,
    'swe1_atp': swe1_atp_settings,
    'swe1_ligand': swe1_ligand_settings,
    'swe1_only': swe1_only_settings,
}

In [ ]:
def loss_nll(S, log_probs, mask):
    """ Negative log probabilities """
    criterion = torch.nn.NLLLoss(reduction='none')
    loss = criterion(
        log_probs.contiguous().view(-1, log_probs.size(-1)), S.contiguous().view(-1)
    ).view(S.size())
    S_argmaxed = torch.argmax(log_probs,-1) #[B, L]
    true_false = (S == S_argmaxed).float()
    loss_masked = loss * mask
    return loss, loss_masked, true_false


In [ ]:
def featurize(
    input_dict,
    cutoff_for_score=8.0,
    use_atom_context=True,
    number_of_ligand_atoms=16,
    model_type="protein_mpnn"):
    output_dict = {}
    if model_type == "ligand_mpnn":
        mask = input_dict["mask"]
        Y = input_dict["Y"]
        Y_t = input_dict["Y_t"]
        Y_m = input_dict["Y_m"]
        N = input_dict["X"][:, 0, :]
        CA = input_dict["X"][:, 1, :]
        C = input_dict["X"][:, 2, :]
        b = CA - N
        c = C - CA
        a = torch.cross(b, c, axis=-1)
        CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA
        Y, Y_t, Y_m, D_XY = get_nearest_neighbours(
            CB, mask, Y, Y_t, Y_m, number_of_ligand_atoms
        )
        mask_XY = (D_XY < cutoff_for_score) * mask * Y_m[:, 0]
        output_dict["mask_XY"] = mask_XY[None,]
        if "side_chain_mask" in list(input_dict):
            output_dict["side_chain_mask"] = input_dict["side_chain_mask"][None,]
        output_dict["Y"] = Y[None,]
        output_dict["Y_t"] = Y_t[None,]
        output_dict["Y_m"] = Y_m[None,]
        if not use_atom_context:
            output_dict["Y_m"] = 0.0 * output_dict["Y_m"]
    elif (
        model_type == "per_residue_label_membrane_mpnn"
        or model_type == "global_label_membrane_mpnn"
    ):
        output_dict["membrane_per_residue_labels"] = input_dict[
            "membrane_per_residue_labels"
        ][None,]

    R_idx_list = []
    count = 0
    R_idx_prev = -100000
    for R_idx in list(input_dict["R_idx"]):
        if R_idx_prev == R_idx:
            count += 1
        R_idx_list.append(R_idx + count)
        R_idx_prev = R_idx
    R_idx_renumbered = torch.tensor(R_idx_list, device=R_idx.device)
    output_dict["R_idx"] = R_idx_renumbered[None,]
    output_dict["R_idx_original"] = input_dict["R_idx"][None,]
    output_dict["chain_labels"] = input_dict["chain_labels"][None,]
    output_dict["S"] = input_dict["S"][None,]
    output_dict["chain_mask"] = input_dict["chain_mask"][None,]
    output_dict["mask"] = input_dict["mask"][None,]

    output_dict["X"] = input_dict["X"][None,]

    if "xyz_37" in list(input_dict):
        output_dict["xyz_37"] = input_dict["xyz_37"][None,]
        output_dict["xyz_37_m"] = input_dict["xyz_37_m"][None,]

    return output_dict

In [7]:
def get_log_probs(input_pdb, chain_list, model, device, ligand_mpnn_use_side_chain_context,
                  parse_atoms_with_zero_occupancy, transmembrane_buried, transmembrane_interface,
                  model_type, ligand_mpnn_cutoff_for_score, ligand_mpnn_use_atom_context, atom_context_num,
                  temperature, updated_alist, nolig=False):
    
    # make protein dict
    protein_dict, backbone, other_atoms, icodes, _ = parse_PDB(
                input_pdb,
                device=device,
                chains=chain_list,
                parse_all_atoms=ligand_mpnn_use_side_chain_context,
                parse_atoms_with_zero_occupancy=parse_atoms_with_zero_occupancy,
                updated_alist=updated_alist,
                modify_list=['3MI3']
            )
    if nolig:
        protein_dict["Y"] = torch.zeros((1, 3), dtype=torch.float32).to(protein_dict["Y"].device)
        protein_dict["Y_t"] = torch.zeros((1), dtype=torch.float32).to(protein_dict["Y_t"].device)
        protein_dict["Y_m"] = torch.zeros((1), dtype=torch.float32).to(protein_dict["Y_m"].device)

    # make chain_letter + residue_idx + insertion_code mapping to integers
    R_idx_list = list(protein_dict["R_idx"].cpu().numpy())  # residue indices
    chain_letters_list = list(protein_dict["chain_letters"])  # chain letters
    encoded_residues = []
    for i, R_idx_item in enumerate(R_idx_list):
        tmp = str(chain_letters_list[i]) + str(R_idx_item) + icodes[i]
        encoded_residues.append(tmp)
    encoded_residue_dict = dict(zip(encoded_residues, range(len(encoded_residues))))
    encoded_residue_dict_rev = dict(
        zip(list(range(len(encoded_residues))), encoded_residues)
    )

    bias_AA_per_residue = torch.zeros(
        [len(encoded_residues), 21], device=device, dtype=torch.float32
    )

    fixed_positions = torch.tensor(
        [int(True) for item in encoded_residues],
        device=device,
    )
    redesigned_positions = torch.tensor(
        [int(False) for item in encoded_residues],
        device=device,
    )

    # specify which residues are buried for checkpoint_per_residue_label_membrane_mpnn model
    if transmembrane_buried:
        buried_residues = [item for item in transmembrane_buried.split()]
        buried_positions = torch.tensor(
            [int(item in buried_residues) for item in encoded_residues],
            device=device,
        )
    else:
        buried_positions = torch.zeros_like(fixed_positions)

    if transmembrane_interface:
        interface_residues = [item for item in transmembrane_interface.split()]
        interface_positions = torch.tensor(
            [int(item in interface_residues) for item in encoded_residues],
            device=device,
        )
    else:
        interface_positions = torch.zeros_like(fixed_positions)
    protein_dict["membrane_per_residue_labels"] = 2 * buried_positions * (
        1 - interface_positions
    ) + 1 * interface_positions * (1 - buried_positions)

    if model_type == "global_label_membrane_mpnn":
        protein_dict["membrane_per_residue_labels"] = (
            args.global_transmembrane_label + 0 * fixed_positions
        )

    # create chain_mask
    chains_to_design_list = protein_dict["chain_letters"]
    chain_mask = torch.tensor(
        np.array(
            [
                item in chains_to_design_list
                for item in protein_dict["chain_letters"]
            ],
            dtype=np.int32,
        ),
        device=device,
    )
    protein_dict["chain_mask"] = chain_mask

    # featurize
    feature_dict = featurize(
        protein_dict,
        cutoff_for_score=ligand_mpnn_cutoff_for_score,
        use_atom_context=ligand_mpnn_use_atom_context,
        number_of_ligand_atoms=atom_context_num,
        model_type=model_type,
    )
    feature_dict["batch_size"] = 1
    feature_dict["temperature"] = temperature
    B, L, _, _ = feature_dict["X"].shape  # batch size should be 1 for now.
    omit_AA = torch.tensor(
        np.array([False for AA in alphabet]).astype(np.float32),
        device=device,
    )
    omit_AA_per_residue = torch.zeros(
        [len(encoded_residues), 21], device=device, dtype=torch.float32
    )
    bias_AA = torch.zeros([21], device=device, dtype=torch.float32)
    bias_AA_per_residue = torch.zeros(
        [len(encoded_residues), 21], device=device, dtype=torch.float32
    )
    feature_dict["bias"] = (
                    (-1e8 * omit_AA[None, None, :] + bias_AA).repeat([1, L, 1])
                    + bias_AA_per_residue[None]
                    - 1e8 * omit_AA_per_residue[None]
                )
    feature_dict["symmetry_residues"] = [[]]
    feature_dict["symmetry_weights"] = [[]]
    feature_dict["randn"] = torch.randn(
        [feature_dict["batch_size"], feature_dict["mask"].shape[1]],
        device=device,
    )

    with torch.no_grad():
        output_dict = model.sample(feature_dict)
        log_probs, etab, E_idx = model(feature_dict)

    _, nlll_loss, nsr = loss_nll(feature_dict["S"].to(dtype=torch.int64), output_dict["log_probs"], feature_dict["mask"])
    
    if etab is not None:
        nlcpl_loss, _ = nlcpl(etab, E_idx, feature_dict["S"].to(dtype=torch.int64), feature_dict["mask"])
        nlcpl_loss = nlcpl_loss.cpu().item()
    else:
        nlcpl_loss = np.nan
        
    return log_probs, nlll_loss, nsr, etab, E_idx, nlcpl_loss

In [ ]:
## Save ProteinMPNN predictions in a DataFrame
def save_preds(mut_sum, mut, pdb, chain, df, row, nlll, nsr):
    col_list = ['ddG_pred', 'mut_type', 'pdb', 'chain', 'nlll', 'nsr']
    if mut_sum != 0:
        val_list = [mut_sum, mut, pdb + '.pdb', chain, nlll, nsr]
        for col, val in zip(col_list, val_list):
            df.loc[row, col] = val

        df.loc[row, 'Model'] = 'ThermoMPNN'
        df.loc[row, 'Dataset'] = 'pdb_test'
    return df

In [ ]:
def get_seq(input_pdb, chain):
    pdb_atoms = parsePDB(input_pdb, chain=chain.upper())
    protein_atoms = pdb_atoms.select("protein")
    try:
        CA_atoms = protein_atoms.select("name CA")
        seq = CA_atoms.getResnames()
        seq = "".join([restype_3to1[AA] if AA in list(restype_3to1) else "X" for AA in list(seq)])
        if '3MI3' in input_pdb:
            seq = seq.strip('K')
    except Exception as e:
        print(input_pdb, chain)
        seq = ""
    
    return seq

In [ ]:
def run_model(pdbs, raw_data_dir, compare_to_exp, test_df, singlechain, offset,
          model, use_potts, ablate_seq, ablate_struct, ablate_nodes, ablate_edges, 
          ablate_struct_layers, ablate_type, ener_fn, model_name, run_name, dataset_name, three_plus_chains=False, model_type='pmpnn',
          split=False, ener_calc='dG', device='cuda:0', ligand_mpnn_use_side_chain_context=0, parse_atoms_with_zero_occupancy=0,
          transmembrane_buried="", transmembrane_interface="", ligand_mpnn_cutoff_for_score=8.0, ligand_mpnn_use_atom_context=1,
          atom_context_num=25, temperature=0.1, updated_alist=True, write_fasta=False):
    
    pdb_save_df = {'pdb': [], 'wt': [], 'mut': [], 'ener': [], 'pred': [], 'pred_mean_norm': []}
    
    ALPHABET = 'ACDEFGHIKLMNPQRSTVWYX'
    pred_dfs = []
    real_dfs = []
    all_corrs = {}
    merge_dfs = {}
    all_nsr = {}
    all_nlll = {}
    all_nlcpl = {}
    all_n = {}
    pearson_dict = {}
    all_refs = []
    all_refs_mean = []
    all_preds = []
    all_preds_mean = []

    for bpdb in tqdm(pdbs):
        pdb = bpdb.split('.')[0]
        if '5UUM_A' in pdb or '5UUM_B' in pdb:
            continue
        input_pdb = os.path.join(raw_data_dir, pdb + '.pdb')
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("structure", input_pdb)
        all_chains = [chain.id for chain in structure.get_chains()]
        if compare_to_exp:
            
            pdb_df = test_df[test_df['WT_name'] == bpdb]
            if len(pdb_df) == 0:
                continue
    #         if len(pdb_df) < 5:
    #             print('dfdd')
    #             continue
    #         if len(pdb_df) < 5:
    #             print('!!!!!!')
            chains = np.unique(pdb_df['chain'].values)
            load_chains = pdb_df['chain_list'].values[0]
            
        else:
            if load_chains == 'all':
                chains = all_chains
                load_chains = chains
            else:
                chains = load_chains
   
        raw_pred_df = pd.DataFrame(columns=['Model', 'Dataset', 'ddG_pred', 'mut_type', 'pdb', 'chain'])
        row = 0
        rcl = 0
        if type(load_chains) == str:
            load_chains = load_chains.split("\'")[1::2]
        for chain in load_chains:
            if compare_to_exp:
                chain_df = pdb_df[pdb_df['chain'] == chain]
                if len(chain_df) > 0:
                    chain_list = chain_df['chain_list'].values[0]
                else:
                    chain_list = load_chains
            elif not singlechain:
                chain_list = load_chains
            if singlechain:
                chain_list = [chain] 
            chain_seq_dict = {}
            if type(chain_list) == str:
                chain_list = chain_list.split("\'")[1::2]
            for chain_id in chain_list:
                chain_seq_dict[chain_id] = get_seq(input_pdb, chain=chain_id)
            chain_seq = chain_seq_dict[chain]
            full_seq_base = "".join([chain_seq_dict[chain_id] for chain_id in chain_list])
            cur_cl = len(chain_seq)
            if chain not in chains:
                rcl += cur_cl
                continue
#             if split:
#                 mut_pdb_target = alt_parse_PDB(input_pdb, ['A'])
#                 mut_pdb_binder = alt_parse_PDB(input_pdb, ['E'])
            if not singlechain:
                chain_lens = [len(chain_seq_dict[c]) for c in chain_list]
#                 try:
#                     chain_lens = [len(chain_seq_dict[c]) for c in chain_list.split("\'")[1::2]]
#                     chain_list = chain_list.split("\'")[1::2]
#                 except:
#                     try:
#                         chain_lens = [len(chain_seq_dict[c]) for c in chain_list]
#                     except:
#                         continue
                        
            mutation_list = get_ssm_mutations(chain_seq)
            final_mutation_list = []

            # build into list of Mutation objects
            for n, m in enumerate(mutation_list):
                if m is None:
                    continue
                m = m.strip()  # clear whitespace
                adj_m = m[0] + str(int(m[1:-1])) + m[-1]
                wtAA, position, mutAA = str(m[0]), int(str(m[1:-1])), str(m[-1])
                assert wtAA in ALPHABET, f"Wild type residue {wtAA} invalid, please try again with one of the following options: {ALPHABET}"
                assert mutAA in ALPHABET, f"Wild type residue {mutAA} invalid, please try again with one of the following options: {ALPHABET}"
                if offset:
                    position += rcl
                if pdb == '4CPA_A_I' or (pdb == '3R9A_A_B_C' and position > 300):
                    position -= 1
                mutation_obj = Mutation(position=position, wildtype=wtAA, mutation=mutAA,
                                        ddG=None, pdb=pdb)
                final_mutation_list.append(mutation_obj)
            rcl += cur_cl
            with torch.no_grad():
                log_probs, nlll_loss, nsr, etab, E_idx, nlcpl_loss = get_log_probs(input_pdb, chain_list, model, device=device,
                                                                       ligand_mpnn_use_side_chain_context=ligand_mpnn_use_side_chain_context,
                                                                       parse_atoms_with_zero_occupancy=parse_atoms_with_zero_occupancy,
                                                                       transmembrane_buried=transmembrane_buried,
                                                                       transmembrane_interface=transmembrane_interface,
                                                                       model_type=model_type,
                                                                       ligand_mpnn_cutoff_for_score=ligand_mpnn_cutoff_for_score,
                                                                       ligand_mpnn_use_atom_context=ligand_mpnn_use_atom_context,
                                                                       atom_context_num=atom_context_num,
                                                                       temperature=temperature,
                                                                       updated_alist=updated_alist
                                                                      )
                if 'split' in run_name:
                    log_probs_nolig, nlll_loss_nolig, nsr_nolig, etab_nolig, E_idx_nolig, nlcpl_loss_nolig = get_log_probs(input_pdb, chain_list, model, device=device,
                                                                       ligand_mpnn_use_side_chain_context=ligand_mpnn_use_side_chain_context,
                                                                       parse_atoms_with_zero_occupancy=parse_atoms_with_zero_occupancy,
                                                                       transmembrane_buried=transmembrane_buried,
                                                                       transmembrane_interface=transmembrane_interface,
                                                                       model_type=model_type,
                                                                       ligand_mpnn_cutoff_for_score=ligand_mpnn_cutoff_for_score,
                                                                       ligand_mpnn_use_atom_context=ligand_mpnn_use_atom_context,
                                                                       atom_context_num=atom_context_num,
                                                                       temperature=temperature,
                                                                       updated_alist=updated_alist,
                                                                       nolig=True
                                                                      )
                
                preds = []
                for mut in final_mutation_list:
                    if mut is None:
                        continue
                    try:
                        pos_log_probs = log_probs[0,mut.position]
                    except Exception as e:
                        print(input_pdb)
                        print(log_probs.shape)
                        print(mut)
                        print(chain_list)
                        print(chain_seq)
                        print(len(chain_seq))
                        print('zo: ', parse_atoms_with_zero_occupancy)
                        protein_dict, backbone, other_atoms, icodes, _ = parse_PDB(
                            input_pdb,
                            device=device,
                            chains=chain_list,
                            parse_all_atoms=ligand_mpnn_use_side_chain_context,
                            parse_atoms_with_zero_occupancy=parse_atoms_with_zero_occupancy,
                        )
                        print("".join(etab_utils.ints_to_seq_torch(protein_dict['S'])))
                        print(protein_dict['S'].shape)
                        atoms = parsePDB(input_pdb)
                        print(input_pdb)
                        print(atoms)
                        raise e
                    aa_index = ALPHABET.index(mut.mutation)
                    wt_aa_index = ALPHABET.index(mut.wildtype)

                    ddg = pos_log_probs[aa_index] - pos_log_probs[wt_aa_index]
                    
                    if 'split' in run_name:
                        ddg = ddg - (log_probs_nolig[0, mut.position][aa_index] - log_probs_nolig[0, mut.position][wt_aa_index])
                    
                    preds.append({
                            "ddG": torch.unsqueeze(ddg, 0),
                            "ddG_ref": mut.ddG
                        })
        
            if nlll_loss is None:
                nlll_loss = np.nan * np.ones(len(preds))
                nsr = np.nan * np.ones(len(preds))
            else:
                nlll_loss = nlll_loss.cpu().numpy()[0]
                nsr = nsr.cpu().numpy()[0]
                if not singlechain and offset:
                    nlll_loss = nlll_loss[rcl - cur_cl:]
                    nsr = nsr[rcl - cur_cl:]
                elif not singlechain and not three_plus_chains: ## POTENTIAL ISSUE!!!
                    nlll_loss = nlll_loss[:chain_lens[0]]
                    nsr = nsr[:chain_lens[0]]
#                 elif three_plus_chains:
#                     rcl = 0
#                     for cid, cl in enumerate(chain_lens):
#                         if chain_list[cid] == chain:
#                             break
#                         rcl += cl
#                     nlll_loss = nlll_loss[rcl:rcl+cl]
#                     nsr = nsr[rcl:rcl+cl]
                        
            all_nsr[pdb] = np.mean(nsr)
            all_nlll[pdb] = np.mean(nlll_loss)
            all_nlcpl[pdb] = nlcpl_loss
            if not compare_to_exp:
                for pred, mut in zip(preds, final_mutation_list):
                    mut_label = mut.wildtype + str(mut.position + 1) + mut.mutation
                    raw_pred_df = save_preds(pred, mut_label, pdb, chain, raw_pred_df, row, nlll_loss[mut.position], nsr[mut.position])
                    row += 1
                continue

            mut_dict = {}
            for pred, mut in zip(preds, final_mutation_list):
                if offset:
                    pos = mut.position - (rcl - cur_cl) #chain_lens[0]
                else:
                    pos = mut.position
                mut_dict[mut.wildtype + str(pos) + mut.mutation] = pred['ddG'].cpu().item()
            if not ('adj_pos' in chain_df.columns):
                chain_df['adj_pos'] = [';'.join(["-1"])] * len(chain_df)
            mut_seqs, mut_eners, pred_eners = [], [], []
            for imut, (mut, adj_pos_list, mut_ener) in enumerate(zip(chain_df['mut_type'].values, chain_df['adj_pos'].values, chain_df['ddG_ML'].values)):
                m_list = mut.split(';')
                adj_pos_list = adj_pos_list.split(';')
                mut_sum = []
                nlll_loss_av = []
                nsr_av = []
                mut_seq = copy.deepcopy(chain_seq_dict[chain])
                if dataset_name == 'ppb':
                    mut_seq = mut_seq.replace('-', '')
                mut_seq = list(mut_seq)
                for indiv_mut, adj_pos in zip(m_list, adj_pos_list):
                    adj_pos = int(adj_pos)
                    if adj_pos < 0:
                        adj_pos = int(indiv_mut[1:-1]) - 1
                    try:
                        assert mut_seq[adj_pos] == indiv_mut[0]
                    except Exception as e:
                        print(mut_seq[adj_pos], indiv_mut[0])
                        print(adj_pos)
                        print("".join(mut_seq))
                        print(pdb)
                        print(indiv_mut)
                        raise ValueError
                    indiv_mut = indiv_mut[0] + str(adj_pos) + indiv_mut[-1]
#                     print(mut_dict)
                    mut_sum.append(mut_dict[indiv_mut])
                    nlll_loss_av.append(nlll_loss[int(indiv_mut[1:-1])])
                    nsr_av.append(nsr[int(indiv_mut[1:-1])])
                    mut_seq[int(indiv_mut[1:-1])] = indiv_mut[-1]
                if len(mut_sum) == 0:
                    continue
                full_seq = copy.deepcopy(full_seq_base.replace('-', ''))
                chain_offset = full_seq.find(chain_seq_dict[chain].replace('-', ''))
                full_seq = list(full_seq)
                
                if write_fasta:
                    imutstr = str(imut)
                    with open(f'/home/fosterb/TERMinator-public_mirror/analysis/ener_inputs/{dataset_name}_{pdb}_{chain}.fasta', 'a') as f:
                        f.write(f'>mut_seq_{imutstr}\n{"".join(mut_seq)}\n')
                full_seq[chain_offset:chain_offset + len(mut_seq)] = mut_seq
                mut_seq = etab_utils.seq_to_ints("".join(full_seq))
                if write_fasta:
                    with open(f'/home/fosterb/TERMinator-public_mirror/analysis/ener_inputs/{dataset_name}_{pdb}.fasta', 'a') as f:
                        f.write(f'>mut_seq_{imutstr}\n{"".join(full_seq)}\n')
                mut_seqs.append(mut_seq)
                mut_eners.append(mut_ener)
                if use_sum: mut_sum = np.sum(mut_sum)
                else: mut_sum = np.mean(mut_sum)
                pred_eners.append(mut_sum)
#                 if mut_sum == 0 and mut[0] != mut[-1]:
#                     print(mut[0], mut[-1])
#                     print(mut_sum, nlll_loss_av, nsr_av)
#                     print(mut, adj_pos_list)
#                     raise ValueError

                raw_pred_df = save_preds(mut_sum, mut, pdb, chain, raw_pred_df, row, np.mean(nlll_loss_av), np.mean(nsr_av))
                row += 1
            if etab is not None and use_potts:
                mut_seqs = torch.from_numpy(np.stack(mut_seqs, axis=0)).unsqueeze(0).to(device=etab.device)
                mut_eners = torch.Tensor(mut_eners).unsqueeze(0).to(device=etab.device)
                data = {
                    'sortcery_seqs': mut_seqs,
                    'sortcery_nrgs': mut_eners,
                    'seqs': torch.Tensor(etab_utils.seq_to_ints(copy.deepcopy(full_seq_base.replace('-', '')))).unsqueeze(0).to(device=etab.device, dtype=torch.int64)
                }
                print(mut_seqs.shape)
                if 'split' in run_name:
                    pearson, predicted_E, mut_eners_post, mut_seqs_post = etab_utils.stability_loss_diff_loop(etab, E_idx, data, etab_nolig, E_idx_nolig,
                                                                                                    return_preds=True,
                                                                                                    max_tokens=100000, return_norm=False, ddg=(ener_calc == 'ddG'))
                else:
                    try:
                        pearson, predicted_E, mut_eners_post, mut_seqs_post = ener_fn(etab, E_idx, data, return_preds=True, return_norm=False)
                    except Exception as e:
                        print(pdb)
                        print(e)
                        continue
                pearson_dict[pdb] = pearson.cpu().item()
                all_preds.append(predicted_E)
                all_preds_mean.append(predicted_E - torch.mean(predicted_E))
                all_refs.append(mut_eners)
                all_refs_mean.append(mut_eners - torch.mean(mut_eners))
                mut_seqs = mut_seqs[0]
                mut_eners = mut_eners[0]
            else:
                try:
                    mut_seqs = torch.from_numpy(np.stack(mut_seqs, axis=0))
                except Exception as e:
                    raise e
                mut_eners = torch.Tensor(mut_eners)
                pred_eners = torch.Tensor(pred_eners)
                all_preds.append(pred_eners)
#                 all_preds.append(torch.from_numpy(np.array(list(raw_pred_df['ddG_pred'].values))))
                all_preds_mean.append(all_preds[-1] - torch.mean(all_preds[-1]))
                all_refs.append(mut_eners)
                all_refs_mean.append(all_refs[-1] - torch.mean(all_refs[-1]))
                            
        # raw_pred_df.to_csv(os.path.join("analysis", f"ProteinMPNN_inference_{pdb}.csv"))
        if compare_to_exp:
            ## Save predictions
            wt_seq = copy.deepcopy(full_seq_base).replace('-', '')
            for mut_seq, mut_ener, mut_pred, mut_pred_mean in zip(mut_seqs, mut_eners, all_preds[-1], all_preds_mean[-1]):
                pdb_save_df['pdb'].append(pdb)
                pdb_save_df['wt'].append(wt_seq)
                mut_seq = "".join(etab_utils.ints_to_seq_torch(mut_seq))
                pdb_save_df['mut'].append(mut_seq)
                pdb_save_df['ener'].append(mut_ener.cpu().item())
                pdb_save_df['pred'].append(mut_pred.cpu().item())
                pdb_save_df['pred_mean_norm'].append(mut_pred_mean.cpu().item())
            pdb_save_df_cur = pd.DataFrame(pdb_save_df)
            pdb_save_df_cur = pdb_save_df_cur[pdb_save_df_cur['pdb'] == pdb]
            if len(pdb_save_df_cur) > 2:
                all_corrs[pdb] = pdb_save_df_cur['ener'].corr(pdb_save_df_cur['pred'])
            else:
                all_corrs[pdb] = np.nan
            all_n[pdb] = len(pdb_save_df_cur)
#             all_corrs[pdb] = 0
                
            raw_pred_df['ddG_pred'] = np.array(list(raw_pred_df['ddG_pred'].values))
            if len(raw_pred_df) >= 0: ## Only run analysis if there are more than 5 experimental values
                raw_pred_df['sig'] = [bpdb + '_' + chain + '_' + mut for pdb, chain, mut in zip(raw_pred_df['pdb'].values, raw_pred_df['chain'].values, raw_pred_df['mut_type'].values)]
                merge_df = pdb_df.merge(raw_pred_df, on='sig', how='inner')
#                 all_corrs[pdb] = merge_df['ddG_pred'].corr(merge_df['ddG_ML'])
                merge_dfs[pdb] = merge_df
                pred_dfs.append(raw_pred_df)
                real_dfs.append(pdb_df)
    pdb_stats_df = {'pdb': [], 'corr': [], 'nsr': [], 'nlll_loss': [], 'nlcpl_loss': [], 'n': []}
    for pdb, corr in all_corrs.items():
        try:
            pdb_stats_df['nsr'].append(all_nsr[pdb])
        except:
            continue
        pdb_stats_df['pdb'].append(pdb)
        pdb_stats_df['corr'].append(corr)
        pdb_stats_df['nlll_loss'].append(all_nlll[pdb])
        pdb_stats_df['nlcpl_loss'].append(all_nlcpl[pdb])
        pdb_stats_df['n'].append(all_n[pdb])
    pdb_stats_df = pd.DataFrame(pdb_stats_df)

    if len(all_refs) > 0:
        all_refs = torch.cat(all_refs)
        all_refs_mean = torch.cat(all_refs_mean)
        all_preds = torch.cat(all_preds)
        all_preds_mean = torch.cat(all_preds_mean)
        
    if compare_to_exp:
        results_dir = f'/home/fosterb/TERMinator-public_mirror/analysis/mpnn_results/{model_name}/'
        os.makedirs(results_dir, exist_ok=True)
        pdb_save_df = pd.DataFrame(pdb_save_df)
        pdb_save_df.to_csv(os.path.join(results_dir, f'{model_name}-{run_name}-{dataset_name}.csv'), index=None)
        pdb_stats_df.to_csv(os.path.join(results_dir, f'{model_name}-{run_name}-{dataset_name}-stats.csv'), index=None)
    return all_refs, all_refs_mean, all_preds, all_preds_mean, pdb_stats_df, all_corrs, merge_dfs, pred_dfs, real_dfs

In [ ]:
element_list = [
        "H",
        "He",
        "Li",
        "Be",
        "B",
        "C",
        "N",
        "O",
        "F",
        "Ne",
        "Na",
        "Mg",
        "Al",
        "Si",
        "P",
        "S",
        "Cl",
        "Ar",
        "K",
        "Ca",
        "Sc",
        "Ti",
        "V",
        "Cr",
        "Mn",
        "Fe",
        "Co",
        "Ni",
        "Cu",
        "Zn",
        "Ga",
        "Ge",
        "As",
        "Se",
        "Br",
        "Kr",
        "Rb",
        "Sr",
        "Y",
        "Zr",
        "Nb",
        "Mb",
        "Tc",
        "Ru",
        "Rh",
        "Pd",
        "Ag",
        "Cd",
        "In",
        "Sn",
        "Sb",
        "Te",
        "I",
        "Xe",
        "Cs",
        "Ba",
        "La",
        "Ce",
        "Pr",
        "Nd",
        "Pm",
        "Sm",
        "Eu",
        "Gd",
        "Tb",
        "Dy",
        "Ho",
        "Er",
        "Tm",
        "Yb",
        "Lu",
        "Hf",
        "Ta",
        "W",
        "Re",
        "Os",
        "Ir",
        "Pt",
        "Au",
        "Hg",
        "Tl",
        "Pb",
        "Bi",
        "Po",
        "At",
        "Rn",
        "Fr",
        "Ra",
        "Ac",
        "Th",
        "Pa",
        "U",
        "Np",
        "Pu",
        "Am",
        "Cm",
        "Bk",
        "Cf",
        "Es",
        "Fm",
        "Md",
        "No",
        "Lr",
        "Rf",
        "Db",
        "Sg",
        "Bh",
        "Hs",
        "Mt",
        "Ds",
        "Rg",
        "Cn",
        "Uut",
        "Fl",
        "Uup",
        "Lv",
        "Uus",
        "Uuo",
        "Unx",
        "X"
    ]
element_list = [item.upper() for item in element_list]